In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
PDF_PATH="Code Simplicity - Max Kanat-Alexander.pdf"
loader=PyPDFLoader(PDF_PATH)
pages=loader.load()
print(f"Loaded {len(pages)} pages from the PDF.")
print(pages[10].page_content[:500])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n","\n","."," "]
)
chunks=splitter.split_documents(pages)
chunks[100]

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store=Chroma.from_documents(chunks, embeddings)
print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 7119.77it/s]


Vector store ready. 389 vectors stored.


In [ ]:
retriever=vector_store.as_retriever(search_kwargs={"k":3})

test_query="How important is testing in clean code?"
retrieved=retriever.invoke(test_query)
for i, doc in enumerate(retrieved, 1):
    print(f"---Chunk {i} ---")
    print(doc.page_content[:300])
    print()

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

# --- Helper: join retrieved chunks into a single  content string ---
def format_docs(docs):
    return "\n\n--\n\n".join(doc.page_content for doc in docs)

SYSTEM_PROMPT="""\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below. 
If the context does not contain enough information, say so clearly.

Context: 
{context}"""

prompt =ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human","{question}"),
])

llm_groq = ChatGroq(model="qwen/qwen3.6-27b",
                    temperature=0,
                    reasoning_format="parsed",
                    reasoning_effort="none")  # we can use reasoning_format too => parsed, raw, hidden

chain=(
    {"context":retriever | format_docs, "question":RunnablePassthrough()}
    | prompt
    |llm_groq
    | StrOutputParser()
)

print("RAG chain assembled")

RAG chain assembled


In [23]:
question = "what about the architecture? is it important?"
print(f"Q: {question} \n")
print("A", chain.invoke(question))

Q: what about the architecture? is it important? 

A Based on the provided context, architecture (or design) is important, but it must be balanced. The text states that "most projects could use more design," implying that sufficient design is beneficial. However, it warns against designing "too much" or going "overboard," comparing excessive design to building an "orbital laser to destroy an anthill." This type of over-engineering is described as costing enormous amounts of money, taking too long to build, and becoming a "maintenance nightmare."

Additionally, the text highlights that good architectural choices—such as building with "simple, self-contained pieces" (like small girders)—make fixing defects and maintaining the system easy, whereas large, complex pieces are difficult to maintain and repair.
